In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q "pyedautils>=0.0.32" ephem statsmodels plotly ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Data Wrangling
This chapter gives an overview of the most important data manipulation functions used throughout this book.

It is a fact that the data import and manipulation part of analyses often takes more time than the actual analysis or visualization itself. This is because the exchange data formats are not standardized and meters and sensors record at different time intervals. Data quality, missing data, data imputation and data cleansing also play a major role.

In [ ]:
# Load data set

import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatTempHum.csv",
                 sep = ";")
df.head()

## Wide to Long

In [ ]:
dfWide = df.copy()
dfWide.head()

In [ ]:
dfLong = dfWide.melt(id_vars=["time"], var_name="dataPoint")
dfLong.head()

### Splitting a column into multiple columns

The `dataPoint` column contains combined information like `FlatA_Hum` — both the flat name and the sensor type. We can split this into two separate columns using `str.split()`:

In [ ]:
# Split "FlatA_Hum" into "flat" = "FlatA" and "sensor" = "Hum"
dfLong[["flat", "sensor"]] = dfLong["dataPoint"].str.split("_", expand=True)
dfLong.head(10)

:::{admonition} How does str.split() work here?
:class: tip dropdown
`"FlatA_Hum".split("_")` splits the string at the underscore and returns `["FlatA", "Hum"]`.

With `expand=True`, pandas creates a DataFrame with one column per part — which we assign directly to two new columns `flat` and `sensor`.

This is useful for filtering, e.g. only temperature sensors:

```python
dfLong[dfLong["sensor"] == "Temp"]
```
:::

In [ ]:
# Remove the original combined column — no longer needed
dfLong = dfLong.drop(columns=["dataPoint"])
dfLong.head()

## Long to Wide

In [ ]:
# Pivot back to wide format using the new flat and sensor columns
dfWide = dfLong.pivot_table(index="time", columns=["flat", "sensor"], values="value")
dfWide.head()

In [ ]:
# Flatten the MultiIndex columns: (FlatA, Hum) -> "FlatA_Hum"
dfWide.columns = [f"{flat}_{sensor}" for flat, sensor in dfWide.columns]
dfWide.head()

:::{admonition} Why flatten MultiIndex columns?
:class: tip dropdown
MultiIndex columns (two levels) make simple operations harder: selecting one column needs a tuple, `df[("FlatA", "Temp")]`, instead of `df["FlatA_Temp"]`.

Flat column names also work better with `.plot()`, `.to_csv()`, and most pandas functions that expect simple string column names.
:::

## Access Data

### Get Values of Column(s)

In [ ]:
df["FlatA_Temp"]

```{note}
A Series is returned!
```

In [ ]:
df[["FlatA_Temp", "FlatA_Hum"]]

```{note}
A DataFrame is returned!
```

## Rename column name(s)

In [ ]:
df.columns

In [ ]:
df1 = df.copy()
df1 = df1.rename({'FlatA_Temp': 'A_Temp', 'FlatA_Hum': 'A_Hum'}, axis=1)
df1.columns

## Select / Drop Column(s)

### Select Column(s) to Keep

In [ ]:
df1 = df[["FlatA_Temp", "FlatB_Temp"]]
df1.head()

### Drop Column(s) by Position

In [ ]:
df2 = df.drop(columns=df.columns[[1, 2]])
df2.head()

### Drop Column(s) by Name

In [ ]:
df3 = df.drop(columns=["FlatA_Hum", "FlatB_Hum", "FlatC_Hum", "FlatD_Hum"])
df3.head()

## Merge / Join

In [ ]:
# Two subsets; Flat B without the hours where its sensor has no reading
df1 = df[["time", "FlatA_Temp", "FlatA_Hum"]]
df2 = df[["time", "FlatB_Temp", "FlatB_Hum"]].dropna()

# Inner join: only times that exist in both DataFrames
df_inner = pd.merge(df1, df2, on="time", how="inner")
print(f"df1: {len(df1)} rows, df2: {len(df2)} rows, inner join: {len(df_inner)} rows")

In [ ]:
# Left join: keep all rows of df1; Flat B columns are NaN where df2 has no row
df_left = pd.merge(df1, df2, on="time", how="left")
print(f"left join: {len(df_left)} rows, {df_left['FlatB_Temp'].isna().sum()} without Flat B values")

## GroupBy

In [ ]:
# Prepare data with datetime index
dfg = df.copy()
dfg["time"] = pd.to_datetime(dfg["time"], format="%Y-%m-%d %H:%M:%S")
dfg = dfg.set_index("time", drop=True)

# Group by month and compute the mean of all columns
dfg.groupby(dfg.index.month).mean()

In [ ]:
# Group by month with multiple aggregation functions
dfg.groupby(dfg.index.month).agg({"FlatA_Temp": ["mean", "min", "max"]})

## Pipe

In [ ]:
# Define simple processing functions
def drop_humidity(df):
    """Drop all humidity columns."""
    hum_cols = [c for c in df.columns if "Hum" in c]
    return df.drop(columns=hum_cols)

def add_temp_mean(df):
    """Add a column with the mean temperature across all flats."""
    temp_cols = [c for c in df.columns if "Temp" in c]
    return df.assign(Temp_Mean=df[temp_cols].mean(axis=1))   # new frame, input unchanged

# Method chaining with .pipe()
dfp = (
    df
    .pipe(drop_humidity)
    .pipe(add_temp_mean)
)
dfp.head()